# clean window check
transaction ids are missing nov 1–11 and jan 26–31 (q6b), so purchases on those days are undercounted.
this reruns the key q1, q5 and q2 results on the clean window only (nov 12 – jan 25, utc) to check the conclusions hold (sql/09_clean_window_check.sql).
r4 rechecks q5b counting any later purchase event, as an upper bound. 
it also splits channels into known vs unknown source and flags small samples (under 30 purchases).


In [1]:
import warnings
warnings.filterwarnings("ignore", message="BigQuery Storage module not found")

from pathlib import Path
import pandas as pd
from google.cloud import bigquery

client = bigquery.Client(project="ga4-funnel-analysis-511719")
output_dir = Path("../outputs")
output_dir.mkdir(exist_ok=True)


In [2]:
sql = """
select
    count(*) as sessions,
    countif(viewed_item or added_to_cart or began_checkout or purchased) as reached_view,
    countif(added_to_cart or began_checkout or purchased) as reached_cart,
    countif(began_checkout or purchased) as reached_checkout,
    countif(purchased) as reached_purchase
from `ga4-funnel-analysis-511719.ga4_analysis.session_base`
where date(timestamp_micros(session_start_time)) between date '2020-11-12' and date '2021-01-25'
"""
wide = client.query(sql).to_dataframe()
wide


,sessions,reached_view,reached_cart,reached_checkout,reached_purchase
0,299173,63280,17784,9160,3968


In [3]:
funnel = wide.melt(var_name="step", value_name="session_count")
funnel["pct_of_previous"] = (funnel["session_count"] / funnel["session_count"].shift(1)).round(4)
funnel["pct_of_all"] = (funnel["session_count"] / funnel["session_count"].iloc[0]).round(4)

full = pd.read_csv(output_dir / "q1_funnel.csv")
funnel = funnel.merge(full[["step", "pct_of_previous", "pct_of_all"]], on="step", suffixes=("", "_full"))
funnel


,step,session_count,pct_of_previous,pct_of_all,pct_of_previous_full,pct_of_all_full
0,sessions,299173,<NA>,1.0,NaN,1.0000
1,reached_view,63280,0.2115,0.2115,0.2147,0.2147
2,reached_cart,17784,0.281,0.0594,0.2630,0.0565
3,reached_checkout,9160,0.5151,0.0306,0.5462,0.0308
4,reached_purchase,3968,0.4332,0.0133,0.3641,0.0112


In [4]:
funnel.to_csv(output_dir / "clean_funnel.csv", index=False)

In [5]:
sql = """
select
    case
        when is_new_visitor then 'new'
        when not is_new_visitor then 'returning'
        else 'unknown'
    end as visitor_type,
    count(*) as sessions,
    countif(viewed_item or added_to_cart or began_checkout or purchased) as reached_view,
    countif(added_to_cart or began_checkout or purchased) as reached_cart,
    countif(began_checkout or purchased) as reached_checkout,
    countif(purchased) as purchases,
    sum(revenue) as revenue
from `ga4-funnel-analysis-511719.ga4_analysis.session_base`
where date(timestamp_micros(session_start_time)) between date '2020-11-12' and date '2021-01-25'
group by visitor_type
order by sessions desc
"""

visitors = client.query(sql).to_dataframe()
visitors["view_rate"] = (visitors["reached_view"] / visitors["sessions"]).round(4)
visitors["cart_rate"] = (visitors["reached_cart"] / visitors["reached_view"]).round(4)
visitors["checkout_rate"] = (visitors["reached_checkout"] / visitors["reached_cart"]).round(4)
visitors["purchase_rate"] = (visitors["purchases"] / visitors["reached_checkout"]).round(4)
visitors["conversion_rate"] = (visitors["purchases"] / visitors["sessions"]).round(4)
visitors["revenue_per_session"] = (visitors["revenue"] / visitors["sessions"]).round(2)
visitors["avg_order_value"] = (visitors["revenue"] / visitors["purchases"]).round(2)
print(visitors["sessions"].sum(), visitors["purchases"].sum(), visitors["revenue"].sum())
visitors


299173 3968 304028.0


,visitor_type,sessions,reached_view,reached_cart,reached_checkout,purchases,revenue,view_rate,cart_rate,checkout_rate,purchase_rate,conversion_rate,revenue_per_session,avg_order_value
0,new,219049,42725,10157,4849,1477,101919.0,0.195,0.2377,0.4774,0.3046,0.0067,0.47,69.0
1,returning,80124,20555,7627,4311,2491,202109.0,0.2565,0.3711,0.5652,0.5778,0.0311,2.52,81.14


In [6]:
visitors.to_csv(output_dir / "clean_new_vs_returning.csv", index=False)

In [7]:
sql = """
select
    case
        when medium is null then 'unknown'
        when source in ('shop.googlemerchandisestore.com', 'googlemerchandisestore.com') then 'self-referral'
        when source = '(direct)' then 'direct'
        when medium = 'organic' then 'organic search'
        when medium = 'cpc' then 'paid search'
        when medium = 'referral' then 'referral'
        when medium = 'email' then 'email'
        when medium = 'affiliate' then 'affiliate'
        when medium in ('<Other>', '(data deleted)') then 'hidden'
        else 'other'
    end as channel,
    count(*) as sessions,
    countif(viewed_item or added_to_cart or began_checkout or purchased) as reached_view,
    countif(purchased) as purchases,
    sum(revenue) as revenue
from `ga4-funnel-analysis-511719.ga4_analysis.session_base`
where date(timestamp_micros(session_start_time)) between date '2020-11-12' and date '2021-01-25'
group by channel
order by sessions desc
"""

channels = client.query(sql).to_dataframe()
channels["view_rate"] = (channels["reached_view"] / channels["sessions"]).round(4)
channels["conversion_rate"] = (channels["purchases"] / channels["sessions"]).round(4)
channels["revenue_per_session"] = (channels["revenue"] / channels["sessions"]).round(2)
channels["source_unknown"] = channels["channel"].isin(["unknown", "self-referral", "hidden"])
channels["small_sample"] = channels["purchases"] < 30
print(channels["sessions"].sum(), channels["purchases"].sum(), channels["revenue"].sum())
channels

299173 3968 304028.0


,channel,sessions,reached_view,purchases,revenue,view_rate,conversion_rate,revenue_per_session,source_unknown,small_sample
0,organic search,83030,21167,1337,94971.0,0.2549,0.0161,1.14,False,False
1,unknown,79276,5482,1,12.0,0.0692,0.0,0.0,True,True
2,self-referral,46672,14669,1167,98556.0,0.3143,0.025,2.11,True,False
3,direct,33063,7445,252,20982.0,0.2252,0.0076,0.63,False,False
4,hidden,27542,6158,264,18601.0,0.2236,0.0096,0.68,True,False
5,referral,21982,6757,884,66198.0,0.3074,0.0402,3.01,False,False
6,paid search,6671,1416,42,2960.0,0.2123,0.0063,0.44,False,False
7,affiliate,850,119,2,167.0,0.14,0.0024,0.2,False,True
8,email,87,67,19,1581.0,0.7701,0.2184,18.17,False,True


In [8]:
channels.to_csv(output_dir / "clean_channels.csv", index=False)

In [9]:
sources = channels.groupby("source_unknown")[["sessions", "purchases", "revenue"]].sum().reset_index()
sources["session_share"] = (sources["sessions"] / sources["sessions"].sum()).round(4)
sources["revenue_share"] = (sources["revenue"] / sources["revenue"].sum()).round(4)
sources["conversion_rate"] = (sources["purchases"] / sources["sessions"]).round(4)
sources


,source_unknown,sessions,purchases,revenue,session_share,revenue_share,conversion_rate
0,False,145683,2536,186859.0,0.487,0.6146,0.0174
1,True,153490,1432,117169.0,0.513,0.3854,0.0093


In [10]:
sources.to_csv(output_dir / "clean_source_unknown.csv", index=False)

In [11]:
sql = """
with first_visits as (
    select
        user_pseudo_id,
        case
            when purchased then 'purchased'
            when began_checkout then 'abandoned checkout'
            when added_to_cart then 'abandoned cart'
            when viewed_item then 'viewed only'
            else 'no product view'
        end as first_visit_result
    from `ga4-funnel-analysis-511719.ga4_analysis.session_base`
    where is_new_visitor
),

later_buyers as (
    select distinct user_pseudo_id
    from `ga4-funnel-analysis-511719.ga4_analysis.session_base`
    where purchased_any
    and not is_new_visitor
)

select
    f.first_visit_result,
    count(*) as users,
    countif(l.user_pseudo_id is not null) as bought_later
from first_visits f
left join later_buyers l
    on f.user_pseudo_id = l.user_pseudo_id
group by first_visit_result
order by users desc
"""

returns_upper = client.query(sql).to_dataframe()
returns_upper["bought_later_rate"] = (returns_upper["bought_later"] / returns_upper["users"]).round(4)
print(returns_upper["users"].sum())
returns_upper

261238


,first_visit_result,users,bought_later,bought_later_rate
0,no product view,209007,539,0.0026
1,viewed only,40588,747,0.0184
2,abandoned cart,5669,338,0.0596
3,abandoned checkout,4468,210,0.047
4,purchased,1506,82,0.0544


In [12]:
returns_upper.to_csv(output_dir / "return_after_first_visit_upper_bound.csv", index=False)